
# Silver Layer for lookup tables
### statustype, tradetype, taxrate, industry

##### Author: Prajwol Regmi

In [0]:
import logging
from pyspark.sql.functions import *
from pyspark.sql.types import *

logger = logging.getLogger("BronzeToSilver_LookupTables")
logger.setLevel(logging.INFO)

In [0]:
%run ../../02_common_utils/operations

In [0]:
dbutils.widgets.text("batch_id", "1")

batch_id = dbutils.widgets.get("batch_id")
run_id = generate_run_id()

bronze_schema = "charles_schwab_retailbrokerage_dev_team_lemma.bronze"
silver_schema = "charles_schwab_retailbrokerage_dev_team_lemma.silver"

results = []

logger.info(f"Pipeline initialized | batch_id={batch_id} | run_id={run_id}")


### statustype

In [0]:
df_statustype = spark.table(f"{bronze_schema}.statustype")
bronze_count = df_statustype.count()

# transformation

df_statustype = df_statustype.select("ST_ID", "ST_NAME") \
    .withColumn("ST_ID", trim(col("ST_ID"))) \
    .withColumn("ST_NAME", trim(col("ST_NAME"))) \
    .filter(col("ST_ID").isNotNull()) \
    .dropDuplicates(["ST_ID"])

# audit columns adding

df_statustype = df_statustype.withColumn("_load_ts", current_timestamp()) \
    .withColumn("_batch", lit(batch_id)) \
    .withColumn("run_id", lit(run_id))

In [0]:
display(df_statustype)

In [0]:
# write & validate

df_statustype.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{silver_schema}.statustype")

# validation

silver_count= spark.table(f"{silver_schema}.statustype").count()
results.append({"table": "statustype", "bronze_count": bronze_count, "silver_count": silver_count, "status": "success" if silver_count > 0 else "empty"})
print(f"statustype: bronze= {bronze_count}, silver= {silver_count}")

### tradetype

In [0]:
df_tradetype = spark.table(f"{bronze_schema}.tradetype")
bronze_count = df_tradetype.count()

#transformation

df_tradetype = df_tradetype.withColumn("TT_ID", trim(col("TT_ID"))) \
    .withColumn("TT_NAME", trim(col("TT_NAME"))) \
    .withColumn("TT_IS_SELL", col("TT_IS_SELL").cast(IntegerType())) \
    .withColumn("TT_IS_MRKT", col("TT_IS_MRKT").cast(IntegerType())) \
    .filter(col("TT_ID").isNotNull()) \
    .dropDuplicates(["TT_ID"])

# add audit columns

df_tradetype = df_tradetype.withColumn("_load_ts", current_timestamp()) \
            .withColumn("_batch", lit(batch_id)) \
            .withColumn("_run_id", lit(run_id))

In [0]:
df_tradetype.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{silver_schema}.tradetype")

# validation

silver_count= spark.table(f"{silver_schema}.tradetype").count()
results.append({"table": "tradetype", "bronze_count": bronze_count, "silver_count": silver_count, "status": "success" if silver_count > 0 else "empty"})

print(f"tradetype: bronze = {bronze_count}, silver= {silver_count}")

## taxrate

In [0]:
df_taxrate = spark.table(f"{bronze_schema}.taxrate")
bronze_count = df_taxrate.count()

# transformation

df_taxrate = df_taxrate.select("TX_ID", "TX_NAME", "TX_RATE") \
        .withColumn("TX_ID", trim(col("TX_ID"))) \
        .withColumn("TX_NAME", trim(col("TX_NAME"))) \
        .withColumn("TX_RATE", col("TX_RATE").cast(DecimalType(6,5))) \
        .filter(col("TX_ID").isNotNull()) \
        .dropDuplicates(["TX_ID"])

# audit columns

df_taxrate = df_taxrate.withColumn("_load_ts", current_timestamp()) \
    .withColumn("_batch", lit(batch_id)) \
    .withColumn("_run_id", lit(run_id))

In [0]:
# write

df_taxrate.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{silver_schema}.taxrate")

silver_count = spark.table(f"{silver_schema}.taxrate").count()
results.append({"table": "taxrate", "bronze_count": bronze_count, "silver_count": silver_count, "status": "success" if silver_count > 0 else "empty"})
print(f"taxrate: bronze={bronze_count}, silver={silver_count}")

## industry

In [0]:
df_industry = spark.table(f"{bronze_schema}.industry")
bronze_count = df_industry.count()

# transformation

df_industry = df_industry.select("IN_ID", "IN_NAME", "IN_SC_ID") \
    .withColumn("IN_ID", trim(col("IN_ID"))) \
    .withColumn("IN_NAME", trim(col("IN_NAME"))) \
    .withColumn("IN_SC_ID", trim(col("IN_SC_ID"))) \
    .filter(col("IN_ID").isNotNull()) \
    .dropDuplicates(["IN_ID"])

# audit

df_industry = df_industry.withColumn("_load_ts", current_timestamp()) \
    .withColumn("_batch", lit(batch_id)) \
    .withColumn("_run_id", lit(run_id))

In [0]:
display(df_industry)

In [0]:
# write
df_industry.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{silver_schema}.industry")


silver_count = spark.table(f"{silver_schema}.industry").count()
results.append({"table": "industry", "bronze_count": bronze_count, "silver_count": silver_count, "status": "SUCCESS" if silver_count > 0 else "EMPTY"})
print(f"industry: bronze={bronze_count}, silver={silver_count}")

In [0]:
recon_df = spark.createDataFrame(results)
display(recon_df)

#audit log

for r in results:
    log_audit_event(spark, run_id, batch_id, "silver", r["table"], "OVERWRITE", r["silver_count"])
    log_pipeline_recon(spark, run_id, batch_id, "cross", r["table"], "bronze", "silver", r["bronze_count"], r["silver_count"])

end_pipeline_run(spark, run_id, "success")
log_domain_run_status(spark, run_id, batch_id, "cross_silver_lookup", "success")

print(f"run id: {run_id}, total tables: {len(results)}")
